# Models Comparison



In [36]:
import plotly.graph_objects as go
import pandas as pd 
import numpy as np

import sys
from pathlib import Path
sys.path.append(str(Path("../../src").resolve())) # or uv sync - but this will always work instead
from pricing_amazon_products.config import (
    MODELS_METADATA_PATH,
    SCORING
)

In [37]:
results_metadata_df = pd.read_parquet(MODELS_METADATA_PATH)

In [38]:
def fmt_num(x, ndigits=4):
    return None if pd.isna(x) else f"{x:.{ndigits}f}"


color_map = {
    "Naive (constant baseline)": "red",
    "ML (basic features)": "orange",
    "ML (basic features and embeddings)": "green",
    "LLM (catalog_content prompt)": "gray"
}

plot_df = results_metadata_df.sort_values("test_score", ascending=False).copy()

fig = go.Figure()

plot_df = results_metadata_df.sort_values("test_score", ascending=False).copy()
feature_order = plot_df["feature_label"].drop_duplicates().tolist()

for feature_label in feature_order:
    group = plot_df[plot_df["feature_label"] == feature_label]

    hover_text = [
        (
            f"Model: {row.model_name}"
            f"<br>Family: {row.model_family}"
            f"<br>Features: {row.feature_label}"
            f"<br>CV RMSE: {fmt_num(row.cv_score_mean)}"
            f" ± {fmt_num(row.cv_score_std)}"
            f"<br>Train RMSE: {fmt_num(row.train_score)}"
            f"<br>Test RMSE: {fmt_num(row.test_score)}"
            f"<br>Notes: {row.params if row.feature_label == 'LLM (catalog_content prompt)' else 'N/A'}"
        )
        for row in group.itertuples(index=False)
    ]

    fig.add_trace(
        go.Bar(
            x=group["model_name"],
            y=group["test_score"],
            name=feature_label,
            marker_color=color_map.get(feature_label, "gray"),
            text=group["test_score"].round(4),
            textposition="outside",
            hovertext=hover_text,
            hoverinfo="text",
        )
    )

fig.update_layout(
    title="Model performance on unseen data",
    yaxis=dict(title="RMSE"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=800,
    barmode="group",
)

fig.show()